# MediFlow 휴대폰 실측 근접 후보 · 추가 1회 학습

**기존 공통 ZIP은 그대로 → 이 새 노트북에서 GPU 선택 → 모두 실행**.
새로운 큰 ZIP 업로드가 필요 없습니다. Drive의 mediflow_Project/MediFlow_Smartphone_WebSkin_Runtime_20261111.zip을 재사용합니다.
이전 1~4차 결과 폴더 `edc193fee3c2`가 Drive에 있어야 합니다. 기존 결과는 읽기만 합니다.

- 출발: 휴대폰 적응 전 선정 PMG·B0·256 저장본. 4차 가중치를 이어 쓰지 않습니다.
- 추가 1회, 5 epoch. Adam 0.00001, batch 8, seed 42, 변환 적용 확률 0.5.
- 밝기 −19.624129908304198~−10.181201425385177, 대비 1~1.1808786167229048.
- 좌우 조명차 −0.42238616943359375~0, Lab a 0~1.0117187500000009, Lab b −1.2126953125000002~0.
- 추가 질감 배율 1, 흐림 σ 0. MMD·CORAL 학습 손실은 사용하지 않습니다.
- 7개 영상 지표가 가까워진 탐색 후보입니다. 조명 불균일·개인차 기준은 미통과이며 실제 장비 동등성이 보장되지 않습니다.
- 기존 Source Train/Validation/Test와 클래스 순서를 유지합니다. Source Test와 P05 사진은 평가하지 않습니다.
- 기존 20개 Validation 결과는 그대로 재사용하고, 새 조건/새 모델 10개를 측정해 6모델×5조건으로 비교합니다.
- 기존 키를 가려진 입력란에 한 번 넣습니다. 설정용 42장만 진단하며 P05 입력은 이 런타임에서 제외합니다.
- 실제 epoch·배치·손실·검증·저장 로그와 학습곡선을 출력합니다. 중단되면 같은 노트북을 다시 실행합니다.

결과는 `MyDrive/MediFlow_smartphone_runs/joint_a3d4036b9f90`와 `MediFlow_Smartphone_Joint_Results_20261112.zip`에 저장됩니다.
실제 휴대폰 질환 정답이 없어 Target Accuracy는 계산하지 않습니다. 자동 패키징·공식 후보 변경은 하지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path, PurePosixPath
import base64, hashlib, io, json, os, subprocess, sys, zipfile
drive.mount("/content/drive")
PROJECT=Path("/content/drive/MyDrive/mediflow_Project")
ZIP_PATH=PROJECT/"MediFlow_Smartphone_WebSkin_Runtime_20261111.zip"
EXPECTED_SHA256="edc193fee3c25254ce947b1a9613ad8ec2cc4d0f90f3c4f283eff5d415ea3ad9"
MANIFEST_SHA256="a3d4036b9f904f4821a0f7cbd0eec111d7a8063617ec0f17493364e403aed984"
def digest(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
if not ZIP_PATH.is_file():raise FileNotFoundError("기존 공통 ZIP이 없습니다: "+str(ZIP_PATH))
print("기존 공통 ZIP 확인 중",flush=True)
if digest(ZIP_PATH)!=EXPECTED_SHA256:raise ValueError("기존 공통 ZIP 버전이 다릅니다. 11월12일 학습에 쓴 파일이 필요합니다.")
RUNTIME=Path("/content/mediflow_phone_joint_a3d4036b9f90")
patch=zipfile.ZipFile(io.BytesIO(base64.b64decode("...")))
frozen=json.loads(patch.read("runtime_manifest.json"))
allowed={*frozen['files'],'runtime_manifest.json'}
if hashlib.sha256(patch.read("runtime_manifest.json")).hexdigest()!=MANIFEST_SHA256:raise ValueError("노트북 내 실행 기록 불일치")
RUNTIME.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as base:
    for name in allowed:
        part=PurePosixPath(name)
        if part.is_absolute() or '..' in part.parts or '\\' in name or ':' in name:raise ValueError("잘못된 런타임 경로")
        path=RUNTIME.joinpath(*part.parts)
        expected=MANIFEST_SHA256 if name=='runtime_manifest.json' else frozen['files'][name]
        if path.exists():
            if digest(path)!=expected:raise ValueError("기존 5차 런타임 파일이 다릅니다: "+name)
            continue
        content=patch.read(name) if name in patch.namelist() else base.read(name)
        if hashlib.sha256(content).hexdigest()!=expected:raise ValueError("실행 파일 해시 불일치: "+name)
        path.parent.mkdir(parents=True,exist_ok=True)
        with path.open('xb') as f:f.write(content)
patch.close()
PREVIOUS=Path("/content/drive/MyDrive/MediFlow_smartphone_runs/edc193fee3c2")
if not PREVIOUS.is_dir():raise FileNotFoundError("이전 결과 폴더를 확인하세요: "+str(PREVIOUS))
OUTPUT=Path("/content/drive/MyDrive/MediFlow_smartphone_runs/joint_a3d4036b9f90")
LOCAL=Path("/content/mediflow_phone_joint_work_a3d4036b9f90")
OUTPUT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(exist_ok=True)
print("추가 학습 결과 저장 위치:",OUTPUT)

In [ ]:
# Colab 기본 TF 패키지를 바꾸지 않는 별도 실행 환경입니다.
VENV = Path("/content/mediflow_phone_env")
PYTHON = VENV/"bin/python"
requirements = ["tensorflow[and-cuda]==2.20.0", "keras==3.13.2", "numpy>=1.26,<3",
                "pillow>=10,<13", "opencv-python-headless>=4.10,<5",
                "matplotlib>=3.8,<4", "cryptography>=44,<49", "tqdm>=4.66,<5"]
probe = "import importlib.metadata as m; assert m.version('tensorflow')=='2.20.0'; assert m.version('keras')=='3.13.2'; import cryptography,cv2,matplotlib,PIL,tqdm"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode or not gpu.stdout.strip():
    raise RuntimeError("Colab 런타임에서 GPU를 선택한 뒤 다시 실행하세요.")
print("GPU:",gpu.stdout.strip())
ready = PYTHON.exists() and subprocess.run([str(PYTHON), "-c", probe], capture_output=True).returncode == 0
if not ready:
    pip_ready = PYTHON.exists() and subprocess.run(
        [str(PYTHON), "-m", "pip", "--version"], capture_output=True).returncode == 0
    if not pip_ready:
        created = subprocess.run([sys.executable, "-m", "venv", str(VENV)], capture_output=True, text=True)
        if created.returncode:
            # Some Colab images omit ensurepip. Bootstrap only the environment creator.
            print("기본 venv 준비를 보완합니다.",flush=True)
            subprocess.run([sys.executable,"-m","pip","install","-q","virtualenv>=20,<21"],check=True)
            subprocess.run([sys.executable,"-m","virtualenv",str(VENV)],check=True)
    subprocess.run([str(PYTHON), "-m", "pip", "install", *requirements], check=True)
ENV = {**os.environ, "PYTHONPATH":str(RUNTIME/"src"), "PYTHONUNBUFFERED":"1",
       "TF_NUM_INTEROP_THREADS":"2", "TF_NUM_INTRAOP_THREADS":"4",
       "TF_CPP_MIN_LOG_LEVEL":"2", "KERAS_BACKEND":"tensorflow",
       "MPLBACKEND":"Agg", "PYTHONDONTWRITEBYTECODE":"1"}
ENV.pop("TF_USE_LEGACY_KERAS", None)
source_code = "from pathlib import Path; from mediflow_datasets.domain_adaptation_batch import find_source; import json; r=Path("+repr(str(RUNTIME))+"); s=json.loads((r/'study.json').read_text()); print(find_source(Path("+repr(str(PROJECT/"datasets"))+"),s['task']))"
found = subprocess.run([str(PYTHON), "-c", source_code], env=ENV, capture_output=True, text=True, check=True)
SOURCE = Path(found.stdout.strip().splitlines()[-1])
if not SOURCE.is_file(): raise FileNotFoundError("기존 web_skin Source ZIP을 확인하세요.")
print("기존 Source 데이터:", SOURCE)

In [ ]:
import getpass, uuid
from datetime import datetime, timezone
key = getpass.getpass("스마트폰 전용 복호화 키 한 줄: ").strip()
if not key: raise ValueError("전용 키 파일 한 줄을 입력하세요.")
log_path = OUTPUT/("training_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]+".log")
command = [str(PYTHON), "-m", "mediflow_datasets.smartphone_joint_training",
           "--runtime",str(RUNTIME), "--source",str(SOURCE),
           "--previous",str(PREVIOUS), "--output",str(OUTPUT), "--local",str(LOCAL)]
# A lock prevents duplicate Run-all cells from touching the same checkpoints.
import fcntl
with (LOCAL/"execution.lock").open("a") as lock:
    try: fcntl.flock(lock, fcntl.LOCK_EX|fcntl.LOCK_NB)
    except BlockingIOError:
        del key
        raise RuntimeError("같은 학습이 이미 실행 중입니다.") from None
    process = None
    try:
        with log_path.open("x",encoding="utf-8") as log:
            process = subprocess.Popen(command,env=ENV,stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            with process.stdin: process.stdin.write(key+"\n")
            del key
            for line in process.stdout:
                print(line,end="",flush=True); log.write(line); log.flush()
            if process.wait(): raise RuntimeError("미완료: 위 로그를 확인하고 같은 노트북을 다시 실행하세요.")
    finally:
        if "key" in globals(): del key
        if process is not None:
            if process.poll() is None: process.terminate(); process.wait()
            process.stdout.close()
print("실행 로그:",log_path)

In [ ]:
from IPython.display import display,Image
summary=json.loads((OUTPUT/"summary.json").read_text())
if summary.get("status")!="completed":raise RuntimeError("추가 학습이 완료되지 않았습니다.")
print(json.dumps(summary,ensure_ascii=False,indent=2))
display(Image(filename=str(OUTPUT/"stage5/learning_curves.png")))
display(Image(filename=str(OUTPUT/"validation_comparison.png")))
display(Image(filename=str(OUTPUT/"stage5_confusion_matrices.png")))
RESULT_ZIP=OUTPUT/"MediFlow_Smartphone_Joint_Results_20261112.zip"
print("Drive 결과 ZIP:",RESULT_ZIP)
from google.colab import files
files.download(str(RESULT_ZIP))